# 09 · Modelos y pipelines

> **Objetivo:** entrenar y comparar los tres modelos del OVA (regresion
> logistica, arbol de profundidad 4, bosque de 300 arboles con hoja ≥ 5)
> contra una **linea base** por reglas; discutir metricas, umbrales,
> importancia y sobreajuste; serializar el modelo final con `metadatos.json`.

## Secuencia (sesion 6 de la bibliografia + sesion 7 adaptada)

1. Metricas desde cero: por que miramos AUC, AP, precision, recall, F1.
2. Linea base: "quien ha comprado recientemente, comprara otra vez".
3. Entrenar los tres modelos con `StratifiedKFold(5, shuffle=True, random_state=42)`.
4. Elegir el umbral de decision (precision/recall segun el costo).
5. Importancia de variables en el bosque.
6. Sobreajuste: curva de profundidad del arbol.
7. Serializar el modelo final con `models/registro.guardar`.

In [ ]:
# Permite importar el paquete segmentacion desde el notebook
import sys
from pathlib import Path

_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [ ]:
from segmentacion.config import FECHA_CORTE_A, FECHA_CORTE_B, HORIZONTE, SEMILLA
from segmentacion.data.cargar import leer_parquet
from segmentacion.data.limpieza import preparar_ventas
from segmentacion.evaluation.metricas import (
    curva_profundidad,
    evaluar_clasificacion,
    regla_recencia,
)
from segmentacion.evaluation.umbral import tabla_umbrales
from segmentacion.features.cohortes import cohorte
from segmentacion.models.entrenar import MODELOS, entrenar
from segmentacion.pipelines.preprocesamiento import CAT, NUM_LOG, NUM_STD, construir_preprocesador

ventas, canc = preparar_ventas(leer_parquet(PROJECT_ROOT / "data" / "raw" / "online_retail_II.parquet"), log=False)
A = cohorte(ventas, canc, FECHA_CORTE_A, horizonte=HORIZONTE)
B = cohorte(ventas, canc, FECHA_CORTE_B, horizonte=HORIZONTE)

cols = NUM_LOG + NUM_STD + CAT
XA, yA = A.select(cols).to_pandas(), A["recompra"].to_numpy()
XB, yB = B.select(cols).to_pandas(), B["recompra"].to_numpy()
print(f"A: {len(XA)} filas | B: {len(XB)} filas")

## 1. Linea base: regla por `recencia`

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

y_regla = regla_recencia(XB)
print(f"acc = {accuracy_score(yB, y_regla):.3f}  P = {precision_score(yB, y_regla):.3f}"
      f"  R = {recall_score(yB, y_regla):.3f}  F1 = {f1_score(yB, y_regla):.3f}")
print(f"baseline mayoritaria (predecir siempre 0): acc = {max(yB.mean(), 1-yB.mean()):.3f}")

El criterio de adopcion escrito antes de entrenar: **cualquier
modelo que no supere a la regla en AUC Y en F1 no justifica el costo
operacional de mantenerlo.** Aqui la regla ya logra 0,629 de F1.

## 2. Entrenar los tres modelos

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline

cv = StratifiedKFold(5, shuffle=True, random_state=SEMILLA)
tabla = {}
for nombre, modelo in MODELOS.items():
    pipe = Pipeline([("pre", construir_preprocesador()), ("mod", modelo)])
    cvres = entrenar(pipe, XA, yA, cv=cv)
    pipe.fit(XA, yA)
    proba = pipe.predict_proba(XB)[:, 1]
    test = evaluar_clasificacion(yB, proba, umbral=0.5)
    tabla[nombre] = {
        "cv_auc": cvres["cv_auc"], "cv_auc_sd": cvres["cv_auc_sd"],
        "test_auc": test["auc"], "test_f1": test["f1"],
        "test_prec": test["precision"], "test_rec": test["recall"],
        "pipe": pipe, "proba": proba,
    }
pd.DataFrame({k: {m: v for m, v in d.items() if m not in ("pipe", "proba")} for k, d in tabla.items()}).T.round(3)

Los tres modelos superan a la regla en AUC (todos > 0,78 vs
0,676 de la regla). El bosque gana por poco contra el arbol en AUC
pero no lo justifica dado su costo. **La regresion logistica es el
mejor balance** para producir una ficha simple e interpretable.

## 3. Umbral de decision

In [ ]:
tabla_umbrales(yB, tabla["Bosque aleatorio"]["proba"], umbrales=(0.3, 0.4, 0.5, 0.6))

En el umbral 0,3 el bosque **señala el doble** de clientes (2 093)
con mejor recall; en 0,6 señala menos pero con precision 0,856. **El
umbral se elige con el equipo de marketing**: no hay un umbral correcto,
hay un balance entre falsos positivos (campaña a quien no la queria) y
falsos negativos (perder a un cliente que si recomprara).

## 4. Importancia de variables (bosque)

In [ ]:
pipe_rf = tabla["Bosque aleatorio"]["pipe"]
nombres = pipe_rf.named_steps["pre"].get_feature_names_out()
importancias = pipe_rf.named_steps["mod"].feature_importances_
pd.DataFrame({"variable": nombres, "importancia": importancias.round(3)}).sort_values("importancia", ascending=False)

`recencia` (0,249), `monetario` (0,218) y `frecuencia` (0,200)
dominan. `pais` tiene importancia ~0,003: Reino Unido no distingue
clientes activos de inactivos, el comportamiento previo si.

## 5. Sobreajuste: curva de profundidad del arbol

In [ ]:
curva = curva_profundidad(XA, yA, XB, yB, construir_preprocesador,
                             profundidades=(1, 2, 3, 4, 6, 8, 12, None))
pd.DataFrame(curva, columns=["profundidad", "AUC(A)", "AUC(B)"])

A profundidad **4** el AUC de B es maximo (0,780); de ahi en
adelante el arbol **memoriza** A (AUC(A) → 1,000) pero su AUC(B) **cae**
(hasta 0,656 sin limite). Es la lectura clasica del sobreajuste.

## 6. Serializar el modelo final para la Unidad 4

In [ ]:
import hashlib
from datetime import datetime

from segmentacion.models.registro import guardar

pipe_final = tabla["Bosque aleatorio"]["pipe"]
hash_datos = hashlib.sha256((PROJECT_ROOT / "data" / "raw" / "online_retail_II.parquet").read_bytes()).hexdigest()[:16]

metadatos = {
    "fecha_corte": str(FECHA_CORTE_A.date()),
    "horizonte": HORIZONTE,
    "variables_en_orden": cols,
    "semilla": SEMILLA,
    "versiones": {"python": "3.11", "polars": "2.0"},  # sklearn se autocompleta
    "datos": {"ruta": "data/raw/online_retail_II.parquet", "hash": hash_datos},
    "auc_validacion": round(tabla["Bosque aleatorio"]["cv_auc"], 3),
    "umbral_acordado": 0.3,
    "fecha": datetime.today().date().isoformat(),
    "autor": "David Palacio (U3)",
}
ruta = PROJECT_ROOT / "models" / "recompra_v1"
guardar(pipe_final, ruta, metadatos)
print(f"artefacto en {ruta}/")
for p in sorted(ruta.iterdir()):
    print(f"  {p.name}: {p.stat().st_size/1024:.1f} KB")

Lo que acabamos de garantizar al equipo de U4:

- La ruta de los datos y su hash permiten reproducir el entrenamiento.
- El orden de las variables de entrada esta congelado.
- La version de scikit-learn se autocompleta y `cargar()` falla con
  mensaje claro si el entorno difiere.
- El umbral acordado (0,3) es parte del contrato, no del codigo de la app.

## Preguntas de reflexion

1. ¿Que pasa si el equipo de marketing decide cambiar el umbral a 0,5?
   ¿Donde se cambia (codigo o metadatos)?
2. ¿Es honesto usar el bosque si la regresion logistica da un AUC
   similar y una ficha mas simple? Argumenta.
3. Si el proximo mes decidimos añadir las 12 columnas del LLM
   (cuaderno 08), ¿como se compara el AUC? ¿Vale la pena mantener el
   LLM en el pipeline? El cuaderno 08 hace esa comparacion una vez.